## METHODS ##

In [2]:
import json
import time
import urllib.request
import pandas as pd
from datetime import datetime
from google import genai
from google.genai import types

ROUTING_SYSTEM_PROMPT = """\
Sei un sistema di routing intelligente per una piattaforma medica.
Il tuo unico compito è analizzare la domanda del medico e restituire in output ESCLUSIVAMENTE un JSON di configurazione.

REGOLE PER I CAMPI DEL JSON:

1. "topics": Lista. Inserisci "pattern_analysis" (per diagnosi, rischi, confronti tra pazienti) e/o "consulto_farmacologico" (per terapie, cure, farmaci).
2. "similarity": Booleano. Metti SEMPRE 'true' se la domanda chiede di guardare al "database", "comorbidità", fare "confronti" o cercare "pazienti simili". Metti 'false' ESCLUSIVAMENTE se chiedono di fare un mero riassunto di chi è il paziente senza cercare connessioni esterne, oppure una domanda guarda al singolo paziente.
3. "k_similar": Intero che indica il numero di pazienti simili da recuperare.
   - PRIORITÀ ASSOLUTA: Se la domanda richiede esplicitamente un numero (es: "confronta 50 pazienti", "trova 30 casi simili"), USA ESATTAMENTE IL NUMERO SPECIFICATO (es. 50, 30), ignorando i default sottostanti.
   - DEFAULT SE NESSUN NUMERO È SPECIFICATO:
     - Usa 5 per richieste banalissime o veloci.
     - Usa 10 COME DEFAULT ASSOLUTO per la quasi totalità delle ricerche mediche.
     - Usa 15 se la richiesta raggiunge livelli estremi di rarità o complessità clinica.
4. "weights_similarity": Array di 3 numeri decimali (somma = 1.0). L'ORDINE DEVE ESSERE ESATTAMENTE QUESTO:
   - Valore 1: Peso Diagnosi
   - Valore 2: Peso Parametri Clinici
   - Valore 3: Peso Dati Demografici
5. "attributes_list": Lista di stringhe.
   - Inserisci SEMPRE: "demographics", "admission_details", "diagnoses", "outcome".
   - SE si chiedono "analisi approfondite"/"complete", OPPURE si menzionano esplicitamente "parametri clinici" o "esami": includi TASSATIVAMENTE anche "vitals", "labs", "previous_diagnoses".
   - SE si affronta il tema di terapie, cure mediche o farmaci: includi "prescriptions".

ESEMPIO STRUTTURALE DI RISPOSTA (da adattare):

{
    "reasoning": "Breve motivazione logica sul perché hai scelto questi parametri.",
    "topics": ["pattern_analysis"],
    "similarity": true,
    "k_similar": 10,
    "weights_similarity": [0.2, 0.2, 0.6],
    "attributes_list": ["demographics", "admission_details", "diagnoses", "outcome", "prescriptions", "vitals", "labs", "previous_diagnoses"]
}
"""

VALID_TOPICS = {"pattern_analysis", "consulto_farmacologico"}
VALID_ATTRS = {"demographics", "admission_details", "vitals", "labs",
               "prescriptions", "diagnoses", "previous_diagnoses", "outcome"}
DEFAULT_WEIGHTS = [0.40, 0.50, 0.10]

def call_orchestrator(question: str, patient_id: int,
                      model_id: str = "gemma4:e2b",
                      api_key: str = None) -> dict:
    """
    Chiama l'orchestratore e restituisce il routing JSON validato.
    Supporta sia Ollama locale (model con ':') sia Gemini API.
    """
    user_prompt = (
        f'Domanda del medico: "{question}"\n'
        f'Paziente target ID (hadm_id): {patient_id}\n\n'
        'Analizza la domanda e genera il JSON di routing.'
    )
    full_prompt = f"{ROUTING_SYSTEM_PROMPT}\n\n{user_prompt}"

    # --- Chiamata LLM ---
    if ":" in model_id:
        # Ollama locale
        payload = json.dumps({
            "model": model_id,
            "prompt": full_prompt,
            "stream": False,
            "format": "json",
            "options": {"temperature": 0.1, "top_p": 0.3, "top_k": 5}
        }).encode("utf-8")
        req = urllib.request.Request(
            "http://localhost:11434/api/generate", data=payload,
            headers={"Content-Type": "application/json"}
        )
        with urllib.request.urlopen(req, timeout=120) as resp:
            result = json.loads(resp.read().decode("utf-8"))
            raw_text = result.get("response", "").strip()
    else:
        # Gemini API
        client = genai.Client(api_key=api_key,
                              http_options=types.HttpOptions(api_version="v1beta"))
        response = client.models.generate_content(
            model=model_id, contents=full_prompt,
            config=types.GenerateContentConfig(temperature=0.1, top_p=0.3, top_k=5)
        )
        raw_text = response.text.strip()

    # --- Parse JSON ---
    if "```json" in raw_text:
        raw_text = raw_text.split("```json")[1].split("```")[0].strip()
    elif "```" in raw_text:
        raw_text = raw_text.split("```")[1].split("```")[0].strip()

    routing = json.loads(raw_text)

    # --- Validazione (stessa logica di _validate_routing) ---
    topics = [t for t in routing.get("topics", []) if t in VALID_TOPICS]
    routing["topics"] = topics if topics else list(VALID_TOPICS)

    sim_val = routing.get("similarity", True)
    if isinstance(sim_val, str):
        sim_val = sim_val.lower() in ("true", "1", "yes")
    routing["similarity"] = bool(sim_val)

    if not routing["similarity"]:
        routing["k_similar"] = 0
        routing["weights_similarity"] = None
    else:
        routing["k_similar"] = int(routing.get("k_similar", 10))
        w = routing.get("weights_similarity")
        if isinstance(w, list) and len(w) == 3 and sum(w) > 0:
            total = sum(w)
            routing["weights_similarity"] = [x / total for x in w]
        else:
            routing["weights_similarity"] = DEFAULT_WEIGHTS

    attrs = [a for a in routing.get("attributes_list", []) if a in VALID_ATTRS]
    routing["attributes_list"] = attrs if attrs else list(VALID_ATTRS)

    return routing

# Ground Truth fields:
#   gt_topics       → topic(s) attesi
#   gt_similarity   → True/False: deve attivare la ricerca simili?
#   gt_has_attrs    → attributi che DEVONO essere presenti nella risposta

CLINICAL_QUESTIONS = [
    # ─── INFO EXTRACTION (Q01-Q10): domande sul singolo paziente, NO similarity ───
    {"id": "Q01", "text": "Qual è l'età e il sesso di questo paziente?",
     "category": "info_extraction",
     "gt_topics": ["pattern_analysis"], "gt_similarity": False,
     "gt_has_attrs": ["demographics"]},

    {"id": "Q02", "text": "Estrai i dati anagrafici principali del paziente dal database.",
     "category": "info_extraction",
     "gt_topics": ["pattern_analysis"], "gt_similarity": False,
     "gt_has_attrs": ["demographics"]},

    {"id": "Q03", "text": "Quali sono le informazioni di base del paziente (età, genere, data di ricovero)?",
     "category": "info_extraction",
     "gt_topics": ["pattern_analysis"], "gt_similarity": False,
     "gt_has_attrs": ["demographics", "admission_details"]},

    {"id": "Q04", "text": "Fornisci un riepilogo dei dati demografici di questo caso.",
     "category": "info_extraction",
     "gt_topics": ["pattern_analysis"], "gt_similarity": False,
     "gt_has_attrs": ["demographics"]},

    {"id": "Q05", "text": "Riassumi le caratteristiche generali del paziente.",
     "category": "info_extraction",
     "gt_topics": ["pattern_analysis"], "gt_similarity": False,
     "gt_has_attrs": ["demographics", "admission_details"]},

    {"id": "Q06", "text": "Quali sono i dettagli di ammissione di questo ricovero?",
     "category": "info_extraction",
     "gt_topics": ["pattern_analysis"], "gt_similarity": False,
     "gt_has_attrs": ["admission_details"]},

    {"id": "Q07", "text": "Descrivi brevemente chi è il paziente in esame.",
     "category": "info_extraction",
     "gt_topics": ["pattern_analysis"], "gt_similarity": False,
     "gt_has_attrs": ["demographics"]},

    {"id": "Q08", "text": "Estrai i dettagli demografici essenziali per questo paziente.",
     "category": "info_extraction",
     "gt_topics": ["pattern_analysis"], "gt_similarity": False,
     "gt_has_attrs": ["demographics"]},

    {"id": "Q09", "text": "Quali informazioni di base abbiamo su questo soggetto?",
     "category": "info_extraction",
     "gt_topics": ["pattern_analysis"], "gt_similarity": False,
     "gt_has_attrs": ["demographics"]},

    {"id": "Q10", "text": "Fai una lista dei dati anagrafici e di ricovero del paziente.",
     "category": "info_extraction",
     "gt_topics": ["pattern_analysis"], "gt_similarity": False,
     "gt_has_attrs": ["demographics", "admission_details"]},

    # ─── COMORBIDITY (Q11-Q20): analisi comorbidità, richiedono similarity ───
    {"id": "Q11", "text": "Quali sono le possibili comorbidità non ancora diagnosticate a questo paziente?",
     "category": "comorbidity",
     "gt_topics": ["pattern_analysis"], "gt_similarity": True,
     "gt_has_attrs": ["diagnoses"]},

    {"id": "Q12", "text": "Analizza le patologie presenti in questo caso clinico.",
     "category": "comorbidity",
     "gt_topics": ["pattern_analysis"], "gt_similarity": False,
     "gt_has_attrs": ["diagnoses"]},

    {"id": "Q13", "text": "Identifica i pattern di comorbidità e confrontali con pazienti simili nel database.",
     "category": "comorbidity",
     "gt_topics": ["pattern_analysis"], "gt_similarity": True,
     "gt_has_attrs": ["diagnoses"]},

    {"id": "Q14", "text": "Fai un'analisi approfondita su quali altre malattie potrebbero insorgere nel paziente.",
     "category": "comorbidity",
     "gt_topics": ["pattern_analysis"], "gt_similarity": True,
     "gt_has_attrs": ["diagnoses", "previous_diagnoses"]},

    {"id": "Q15", "text": "Descrivi il quadro generale delle diagnosi attribuite al paziente durante il ricovero.",
     "category": "comorbidity",
     "gt_topics": ["pattern_analysis"], "gt_similarity": False,
     "gt_has_attrs": ["diagnoses"]},

    {"id": "Q16", "text": "Esistono correlazioni tra le diverse diagnosi di questo paziente e quelle di casi simili?",
     "category": "comorbidity",
     "gt_topics": ["pattern_analysis"], "gt_similarity": True,
     "gt_has_attrs": ["diagnoses"]},

    {"id": "Q17", "text": "Qual è la percentuale di decesso dei pazienti simili?",
     "category": "comorbidity",
     "gt_topics": ["pattern_analysis"], "gt_similarity": True,
     "gt_has_attrs": ["outcome"]},

    {"id": "Q18", "text": "Elenca le condizioni cliniche secondarie che potrebbero influenzare il quadro del paziente.",
     "category": "comorbidity",
     "gt_topics": ["pattern_analysis"], "gt_similarity": True,
     "gt_has_attrs": ["diagnoses"]},

    {"id": "Q19", "text": "Analizza la complessità clinica del paziente basandoti sulle comorbidità e confronta con il database.",
     "category": "comorbidity",
     "gt_topics": ["pattern_analysis"], "gt_similarity": True,
     "gt_has_attrs": ["diagnoses"]},

    {"id": "Q20", "text": "Quali pattern diagnostici secondari emergono dalle cartelle cliniche?",
     "category": "comorbidity",
     "gt_topics": ["pattern_analysis"], "gt_similarity": True,
     "gt_has_attrs": ["diagnoses"]},

    # ─── PHARMACOLOGY (Q21-Q30): consigli farmacologici ───
    {"id": "Q21", "text": "Quali consigli farmacologici daresti per questo paziente?",
     "category": "pharmacology",
     "gt_topics": ["consulto_farmacologico"], "gt_similarity": True,
     "gt_has_attrs": ["prescriptions", "diagnoses"]},

    {"id": "Q22", "text": "Qual è la terapia farmacologica ottimale basata sulla storia clinica?",
     "category": "pharmacology",
     "gt_topics": ["consulto_farmacologico"], "gt_similarity": True,
     "gt_has_attrs": ["prescriptions", "diagnoses"]},

    {"id": "Q23", "text": "Consiglia dei farmaci appropriati per trattare le patologie di questo paziente.",
     "category": "pharmacology",
     "gt_topics": ["consulto_farmacologico"], "gt_similarity": True,
     "gt_has_attrs": ["prescriptions", "diagnoses"]},

    {"id": "Q24", "text": "Quali sono le raccomandazioni terapeutiche per questo specifico quadro clinico?",
     "category": "pharmacology",
     "gt_topics": ["consulto_farmacologico"], "gt_similarity": True,
     "gt_has_attrs": ["prescriptions", "diagnoses"]},

    {"id": "Q25", "text": "Fornisci suggerimenti sull'uso dei farmaci per questo paziente.",
     "category": "pharmacology",
     "gt_topics": ["consulto_farmacologico"], "gt_similarity": True,
     "gt_has_attrs": ["prescriptions"]},

    {"id": "Q26", "text": "Quale regime farmacologico sarebbe più indicato per questo caso?",
     "category": "pharmacology",
     "gt_topics": ["consulto_farmacologico"], "gt_similarity": True,
     "gt_has_attrs": ["prescriptions", "diagnoses"]},

    {"id": "Q27", "text": "Ci sono interazioni farmacologiche da evitare per questo paziente?",
     "category": "pharmacology",
     "gt_topics": ["consulto_farmacologico"], "gt_similarity": True,
     "gt_has_attrs": ["prescriptions"]},

    {"id": "Q28", "text": "Indica una possibile strategia farmacologica per migliorare l'esito clinico.",
     "category": "pharmacology",
     "gt_topics": ["consulto_farmacologico"], "gt_similarity": True,
     "gt_has_attrs": ["prescriptions", "outcome"]},

    {"id": "Q29", "text": "Quali farmaci dovrebbero essere considerati prioritari per questo paziente?",
     "category": "pharmacology",
     "gt_topics": ["consulto_farmacologico"], "gt_similarity": False,
     "gt_has_attrs": ["prescriptions"]},

    {"id": "Q30", "text": "Sulla base delle diagnosi, quali sono i principali consigli terapeutici?",
     "category": "pharmacology",
     "gt_topics": ["consulto_farmacologico"], "gt_similarity": False,
     "gt_has_attrs": ["prescriptions", "diagnoses"]},

    # ─── SIMILARITY / CONFRONTI (Q31-Q40): richiedono similarity = True ───
    {"id": "Q31", "text": "Confronta questo paziente con casi simili nel database per identificare pattern comuni.",
     "category": "similarity",
     "gt_topics": ["pattern_analysis"], "gt_similarity": True,
     "gt_has_attrs": ["diagnoses"]},

    {"id": "Q32", "text": "Come si posiziona questo paziente rispetto a soggetti con diagnosi analoghe?",
     "category": "similarity",
     "gt_topics": ["pattern_analysis"], "gt_similarity": True,
     "gt_has_attrs": ["diagnoses"]},

    {"id": "Q33", "text": "Analizza le differenze e le somiglianze tra questo paziente e i suoi simili.",
     "category": "similarity",
     "gt_topics": ["pattern_analysis"], "gt_similarity": True,
     "gt_has_attrs": ["diagnoses"]},

    {"id": "Q34", "text": "Quali sono i risultati clinici tipici per pazienti simili a questo?",
     "category": "similarity",
     "gt_topics": ["pattern_analysis"], "gt_similarity": True,
     "gt_has_attrs": ["outcome", "diagnoses"]},

    {"id": "Q35", "text": "Confronta il decorso di questo paziente con quello di altri casi comparabili.",
     "category": "similarity",
     "gt_topics": ["pattern_analysis"], "gt_similarity": True,
     "gt_has_attrs": ["outcome"]},

    {"id": "Q36", "text": "Esistono pazienti nel database con un quadro clinico quasi identico?",
     "category": "similarity",
     "gt_topics": ["pattern_analysis"], "gt_similarity": True,
     "gt_has_attrs": ["diagnoses"]},

    {"id": "Q37", "text": "Usa i dati dei pazienti simili per valutare la tipicità di questo caso.",
     "category": "similarity",
     "gt_topics": ["pattern_analysis"], "gt_similarity": True,
     "gt_has_attrs": ["diagnoses"]},

    {"id": "Q38", "text": "In che modo questo paziente si discosta dai pattern standard di pazienti simili?",
     "category": "similarity",
     "gt_topics": ["pattern_analysis"], "gt_similarity": True,
     "gt_has_attrs": ["diagnoses"]},

    {"id": "Q39", "text": "Quali strategie terapeutiche hanno funzionato meglio in pazienti simili a questo?",
     "category": "similarity",
     "gt_topics": ["pattern_analysis", "consulto_farmacologico"], "gt_similarity": True,
     "gt_has_attrs": ["prescriptions", "diagnoses"]},

    {"id": "Q40", "text": "Esegui un'analisi comparativa tra questo paziente e la popolazione simile nel database.",
     "category": "similarity",
     "gt_topics": ["pattern_analysis"], "gt_similarity": True,
     "gt_has_attrs": ["diagnoses"]},

    # ─── MIXED (Q41-Q50): combinazioni di categorie ───
    {"id": "Q41", "text": "Riassumi le terapie e le comorbidità di questo paziente.",
     "category": "mixed",
     "gt_topics": ["pattern_analysis", "consulto_farmacologico"], "gt_similarity": False,
     "gt_has_attrs": ["diagnoses", "prescriptions"]},

    {"id": "Q42", "text": "Cosa suggeriscono i dati dei pazienti simili riguardo al trattamento di questo caso?",
     "category": "mixed",
     "gt_topics": ["consulto_farmacologico"], "gt_similarity": True,
     "gt_has_attrs": ["prescriptions"]},

    {"id": "Q43", "text": "Estrai info su farmaci e diagnosi e confrontale con la media del database.",
     "category": "mixed",
     "gt_topics": ["pattern_analysis", "consulto_farmacologico"], "gt_similarity": True,
     "gt_has_attrs": ["prescriptions", "diagnoses"]},

    {"id": "Q44", "text": "Quali sono i punti critici del quadro clinico e farmacologico di questo paziente?",
     "category": "mixed",
     "gt_topics": ["pattern_analysis", "consulto_farmacologico"], "gt_similarity": True,
     "gt_has_attrs": ["diagnoses", "prescriptions"]},

    {"id": "Q45", "text": "Fornisci un riassunto approfondito di tutto ciò che sai sul paziente.",
     "category": "mixed",
     "gt_topics": ["pattern_analysis", "consulto_farmacologico"], "gt_similarity": False,
     "gt_has_attrs": ["demographics", "diagnoses", "vitals", "prescriptions"]},

    {"id": "Q46", "text": "Analizza le patologie del paziente e consiglia una terapia basata su casi analoghi.",
     "category": "mixed",
     "gt_topics": ["pattern_analysis", "consulto_farmacologico"], "gt_similarity": True,
     "gt_has_attrs": ["diagnoses", "prescriptions"]},

    {"id": "Q47", "text": "Quali sono i dati salienti del paziente e quali pattern di rischio emergono dal database?",
     "category": "mixed",
     "gt_topics": ["pattern_analysis"], "gt_similarity": True,
     "gt_has_attrs": ["demographics", "diagnoses", "outcome"]},

    {"id": "Q48", "text": "Analizza i parametri di laboratorio e i segni vitali critici di questo paziente confrontandoli con pazienti simili",
     "category": "mixed",
     "gt_topics": ["pattern_analysis"], "gt_similarity": True,
     "gt_has_attrs": ["vitals", "labs"]},

    {"id": "Q49", "text": "Confronta il profilo di rischio di questo paziente con casi clinici affini nel database.",
     "category": "mixed",
     "gt_topics": ["pattern_analysis"], "gt_similarity": True,
     "gt_has_attrs": ["diagnoses", "outcome"]},

    {"id": "Q50", "text": "Quali sono le informazioni chiave da considerare per la gestione complessiva di questo paziente?",
     "category": "mixed",
     "gt_topics": ["pattern_analysis"], "gt_similarity": True,
     "gt_has_attrs": ["demographics", "admission_details", "diagnoses", "outcome"]},
]

def evaluate_orchestrator(questions: list,
                          model_id: str = "gemma4:e2b",
                          api_key: str = None,
                          patient_id: int = 26133978,
                          delay: float = 0.5) -> pd.DataFrame:
    """
    Esegue l'orchestratore su ogni prompt e confronta l'output col ground truth.
    
    Metriche calcolate per ogni prompt:
      - topics_match   : i topic predetti contengono tutti quelli attesi?
      - sim_match      : similarity predetto == ground truth?
      - attrs_match    : tutti gli attributi attesi sono presenti?
      - score          : media delle 3 metriche (0-1)
    """
    results = []
    total = len(questions)
    
    print(f"{'='*60}")
    print(f"  ORCHESTRATOR EVALUATION — {total} prompts")
    print(f"  Model: {model_id} | Patient ID: {patient_id}")
    print(f"{'='*60}\n")

    for i, q in enumerate(questions, 1):
        print(f"  [{i:02d}/{total}] {q['id']} ({q['category']}): {q['text'][:55]}...", end="")
        
        try:
            t0 = time.perf_counter()
            routing = call_orchestrator(q["text"], patient_id, model_id, api_key)
            elapsed = time.perf_counter() - t0
            
            # --- Confronto con Ground Truth ---
            pred_topics = set(routing.get("topics", []))
            gt_topics   = set(q["gt_topics"])
            topics_match = gt_topics.issubset(pred_topics)

            pred_sim = routing.get("similarity", False)
            gt_sim   = q["gt_similarity"]
            sim_match = (pred_sim == gt_sim)

            pred_attrs = set(routing.get("attributes_list", []))
            gt_attrs   = set(q["gt_has_attrs"])
            attrs_match = gt_attrs.issubset(pred_attrs)

            score = sum([topics_match, sim_match, attrs_match]) / 3.0

            status_icon = "✅" if score == 1.0 else ("⚠️" if score >= 0.66 else "❌")
            print(f"  {status_icon} {elapsed:.1f}s | score={score:.2f}")

            # Log errori specifici
            if not topics_match:
                print(f"       ↳ Topics: atteso {q['gt_topics']}, ottenuto {list(pred_topics)}")
            if not sim_match:
                print(f"       ↳ Similarity: atteso {gt_sim}, ottenuto {pred_sim}")
            if not attrs_match:
                missing = gt_attrs - pred_attrs
                print(f"       ↳ Attrs mancanti: {missing}")

            results.append({
                "id": q["id"],
                "category": q["category"],
                "question": q["text"],
                # Predizioni
                "pred_topics": list(pred_topics),
                "pred_similarity": pred_sim,
                "pred_k_similar": routing.get("k_similar", 0),
                "pred_attrs": list(pred_attrs),
                "pred_reasoning": routing.get("reasoning", ""),
                # Ground Truth
                "gt_topics": q["gt_topics"],
                "gt_similarity": gt_sim,
                "gt_has_attrs": q["gt_has_attrs"],
                # Metriche
                "topics_ok": topics_match,
                "sim_ok": sim_match,
                "attrs_ok": attrs_match,
                "score": score,
                "time_s": round(elapsed, 2),
                "status": "success",
            })

        except Exception as e:
            print(f"  ❌ ERRORE: {e}")
            results.append({
                "id": q["id"], "category": q["category"], "question": q["text"],
                "pred_topics": [], "pred_similarity": None, "pred_k_similar": None,
                "pred_attrs": [], "pred_reasoning": "",
                "gt_topics": q["gt_topics"], "gt_similarity": q["gt_similarity"],
                "gt_has_attrs": q["gt_has_attrs"],
                "topics_ok": False, "sim_ok": False, "attrs_ok": False,
                "score": 0.0, "time_s": 0, "status": f"error: {e}",
            })

        time.sleep(delay)

    return pd.DataFrame(results)

'''
# Avvia la valutazione (Ollama locale, nessuna API key necessaria)
df = evaluate_orchestrator(
    questions=CLINICAL_QUESTIONS,
    model_id="gemma4:e2b",       # il tuo modello Ollama
    patient_id=26133978,          # un hadm_id qualsiasi
    delay=0.3,
)

# --- Report Aggregato ---
print(f"\n{'='*60}")
print(f"  REPORT FINALE")
print(f"{'='*60}")

success_df = df[df["status"] == "success"]
n = len(success_df)

print(f"\n  Prompt eseguiti con successo: {n}/{len(df)}")
print(f"  Tempo medio per prompt:      {success_df['time_s'].mean():.2f}s")
print(f"\n  --- ACCURACY PER METRICA ---")
print(f"  Topics Match:     {success_df['topics_ok'].mean()*100:.1f}%")
print(f"  Similarity Match: {success_df['sim_ok'].mean()*100:.1f}%")
print(f"  Attributes Match: {success_df['attrs_ok'].mean()*100:.1f}%")
print(f"  Score Medio:      {success_df['score'].mean()*100:.1f}%")

# Breakdown per categoria
print(f"\n  --- ACCURACY PER CATEGORIA ---")
cat_stats = success_df.groupby("category").agg(
    n=("score", "count"),
    topics_acc=("topics_ok", "mean"),
    sim_acc=("sim_ok", "mean"),
    attrs_acc=("attrs_ok", "mean"),
    avg_score=("score", "mean"),
).round(3)
cat_stats[["topics_acc", "sim_acc", "attrs_acc", "avg_score"]] *= 100
print(cat_stats.to_string())

# Salva CSV
output_csv = f"orchestrator_eval_{datetime.now().strftime('%Y%m%d_%H%M%S')}.csv"
df.to_csv(output_csv, index=False)
print(f"\n  📁 Risultati salvati in: {output_csv}")

'''


'\n# Avvia la valutazione (Ollama locale, nessuna API key necessaria)\ndf = evaluate_orchestrator(\n    questions=CLINICAL_QUESTIONS,\n    model_id="gemma4:e2b",       # il tuo modello Ollama\n    patient_id=26133978,          # un hadm_id qualsiasi\n    delay=0.3,\n)\n\n# --- Report Aggregato ---\nprint(f"\n{\'=\'*60}")\nprint(f"  REPORT FINALE")\nprint(f"{\'=\'*60}")\n\nsuccess_df = df[df["status"] == "success"]\nn = len(success_df)\n\nprint(f"\n  Prompt eseguiti con successo: {n}/{len(df)}")\nprint(f"  Tempo medio per prompt:      {success_df[\'time_s\'].mean():.2f}s")\nprint(f"\n  --- ACCURACY PER METRICA ---")\nprint(f"  Topics Match:     {success_df[\'topics_ok\'].mean()*100:.1f}%")\nprint(f"  Similarity Match: {success_df[\'sim_ok\'].mean()*100:.1f}%")\nprint(f"  Attributes Match: {success_df[\'attrs_ok\'].mean()*100:.1f}%")\nprint(f"  Score Medio:      {success_df[\'score\'].mean()*100:.1f}%")\n\n# Breakdown per categoria\nprint(f"\n  --- ACCURACY PER CATEGORIA ---")\ncat_stats

## ORCHESTRATOR TEST ##

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# CELLA 3 — Valutazione Integrata: 50 Pazienti × 1 Domanda
#
# Sequenza per ogni coppia (hadm_id, domanda):
#   1) Chiama l'orchestratore → routing JSON
#   2) Se routing["similarity"] == True:
#        usa routing["k_similar"] e routing["weights_similarity"]
#        per chiamare SimilarityEngine.find_similar()
#        e calcola Hit-Rate / MRR (Jaccard)
#   3) Registra i risultati di entrambi i test
#
# PREREQUISITI: le celle precedenti devono essere state eseguite:
#   → definiscono CLINICAL_QUESTIONS, call_orchestrator,
#     VALID_TOPICS, VALID_ATTRS, DEFAULT_WEIGHTS
# ═══════════════════════════════════════════════════════════════════
import sys
import ast
import time
import logging
import pandas as pd
import numpy as np
from pathlib import Path
from datetime import datetime
from tqdm.auto import tqdm

# ── 0. Path setup — usa la copia locale degli agents ───────────────
AI_AGENTIC_PATH = Path("../ai-agentic").resolve()
RAGAS_PATH      = Path(".").resolve()  # cartella corrente (ai-agentic-ragas)

# Aggiungi prima la cartella ragas (ha agents/ e config/ locali)
for p in [str(RAGAS_PATH), str(AI_AGENTIC_PATH)]:
    if p not in sys.path:
        sys.path.insert(0, p)

from config.settings import Settings
from agents.similarity_engine import SimilarityEngine

# Silenzia logger interni dell'engine
for _lg in ("similarity_engine", "timer", "data_gatherer"):
    logging.getLogger(_lg).setLevel(logging.WARNING)

# ── 1. Parametri ───────────────────────────────────────────────────
CHECKPOINT_DIR    = AI_AGENTIC_PATH / "data" / "pipeline_checkpoints"
PARQUET_PATH      = str(CHECKPOINT_DIR / "combined_data_slim.parquet")
N_PATIENTS        = 50
RANDOM_SEED       = 42
ORCHESTRATOR_MODEL = "gemma4:e2b"   # modello Ollama locale
ORCHESTRATOR_DELAY = 0.3
# Parametri similarity (usati come fallback se il routing non li fornisce)
DEFAULT_SIM_K         = 5
DEFAULT_SIM_WEIGHTS   = [0.60, 0.20, 0.20]  # [diag, numeric, categorical]
JACCARD_THRESHOLD     = 0.2
USE_CATEGORIES        = True   # True = categorie ICD, False = codici esatti

# ── 2. Campiona 50 hadm_id casuali dal parquet ─────────────────────
print("Caricamento hadm_id dal parquet...")
df_ids = pd.read_parquet(PARQUET_PATH, columns=["hadm_id"])
all_hadm_ids = df_ids["hadm_id"].to_numpy()

rng = np.random.default_rng(RANDOM_SEED)
sample_patients = rng.choice(all_hadm_ids, size=N_PATIENTS, replace=False).tolist()
print(f"✅ Campionati {N_PATIENTS} pazienti (seed={RANDOM_SEED})")
print(f"   Esempio ID: {[int(x) for x in sample_patients[:5]]}\n")

# Assegna una domanda casuale a ciascun paziente (shuffle delle 50 domande)
rng2 = np.random.default_rng(RANDOM_SEED + 1)
question_order = rng2.permutation(len(CLINICAL_QUESTIONS)).tolist()
assigned_questions = [CLINICAL_QUESTIONS[question_order[i]] for i in range(N_PATIENTS)]

# ── 3. Inizializza SimilarityEngine UNA SOLA VOLTA ────────────────
print("Caricamento ground truth diagnosi dal parquet...")
col_name = "diagnosis_categories_list" if USE_CATEGORIES else "diagnoses"
df_diag  = pd.read_parquet(PARQUET_PATH, columns=["hadm_id", col_name])

diag_dict: dict = {}
for _, row in df_diag.iterrows():
    val = row[col_name]
    hid = int(row["hadm_id"])
    
    # Estraiamo i codici e tagliamoli alla terza cifra
    raw_set = set()
    if isinstance(val, (list, np.ndarray)):
        raw_set = set(val)
    elif isinstance(val, str):
        try:
            raw_set = set(ast.literal_eval(val))
        except Exception:
            raw_set = set(val.split(","))
    
    # AGGIUNGI QUESTA RIGA: Tronca ogni codice a 3 caratteri
    diag_dict[hid] = {str(code)[:3] for code in raw_set if code}
print(f"✅ Ground truth caricata per {len(diag_dict)} pazienti")

# hadm_to_idx: mappa hadm_id → indice posizionale (ordine nel parquet = ordine .npy)
hadm_to_idx = {int(hid): i for i, hid in enumerate(all_hadm_ids)}

settings = Settings()
settings.checkpoint_path = str(CHECKPOINT_DIR)
engine = SimilarityEngine(settings)
print("Connessione a Milvus e caricamento embedding...")
engine.load(hadm_to_idx)
print("✅ SimilarityEngine pronto\n")

# ── 4. Loop principale: 1 paziente → 1 domanda ────────────────────
print("=" * 70)
print(f"  VALUTAZIONE INTEGRATA — {N_PATIENTS} coppie (paziente, domanda)")
print("=" * 70)

rows = []

for i, (patient_id, question) in enumerate(
    zip(sample_patients, assigned_questions), start=1
):
    pat_int = int(patient_id)
    qid     = question["id"]
    qcat    = question["category"]
    qtext   = question["text"]

    print(f"\n  [{i:02d}/{N_PATIENTS}] {qid} ({qcat}) | hadm_id={pat_int}")
    print(f"           Q: {qtext[:65]}...")

    row = {
        "patient_idx":    i,
        "hadm_id":        pat_int,
        "q_id":           qid,
        "q_category":     qcat,
        "q_text":         qtext,
        "gt_similarity":  question["gt_similarity"],
        "gt_topics":      question["gt_topics"],
        "gt_has_attrs":   question["gt_has_attrs"],
    }

    # ── A) ORCHESTRATORE ──────────────────────────────────────────
    try:
        t0 = time.perf_counter()
        routing = call_orchestrator(qtext, pat_int, ORCHESTRATOR_MODEL)
        orch_elapsed = time.perf_counter() - t0

        pred_topics = set(routing.get("topics", []))
        gt_topics   = set(question["gt_topics"])
        topics_ok   = gt_topics.issubset(pred_topics)

        pred_sim = routing.get("similarity", False)
        gt_sim   = question["gt_similarity"]
        sim_ok   = (pred_sim == gt_sim)

        pred_attrs = set(routing.get("attributes_list", []))
        gt_attrs   = set(question["gt_has_attrs"])
        attrs_ok   = gt_attrs.issubset(pred_attrs)

        orch_score = (topics_ok + sim_ok + attrs_ok) / 3.0
        icon = "✅" if orch_score == 1.0 else ("⚠️" if orch_score >= 0.66 else "❌")

        print(f"           Orchestratore: {icon} {orch_elapsed:.1f}s | score={orch_score:.2f}", end="")
        if not topics_ok:
            print(f" | topics attesi={question['gt_topics']} ottenuti={list(pred_topics)}", end="")
        if not sim_ok:
            print(f" | sim atteso={gt_sim} ottenuto={pred_sim}", end="")
        if not attrs_ok:
            print(f" | attrs mancanti={gt_attrs - pred_attrs}", end="")
        print()

        row.update({
            "pred_topics":     list(pred_topics),
            "pred_similarity": pred_sim,
            "pred_k_similar":  routing.get("k_similar", 0),
            "pred_weights":    routing.get("weights_similarity"),
            "pred_attrs":      list(pred_attrs),
            "pred_reasoning":  routing.get("reasoning", ""),
            "topics_ok":       topics_ok,
            "sim_ok":          sim_ok,
            "attrs_ok":        attrs_ok,
            "orch_score":      orch_score,
            "orch_time_s":     round(orch_elapsed, 2),
            "orch_status":     "success",
        })

    except Exception as e:
        print(f"           Orchestratore: ❌ ERRORE: {e}")
        row.update({
            "pred_topics": [], "pred_similarity": None,
            "pred_k_similar": None, "pred_weights": None,
            "pred_attrs": [], "pred_reasoning": "",
            "topics_ok": False, "sim_ok": False, "attrs_ok": False,
            "orch_score": 0.0, "orch_time_s": 0.0,
            "orch_status": f"error: {e}",
        })
        # Senza routing valido salta la similarity
        row.update({
            "sim_performed": False, "sim_hit": None,
            "sim_mrr": None, "sim_best_jaccard": None,
            "sim_k_used": None, "sim_weights_used": None,
            "sim_status": "skipped (orch error)",
        })
        rows.append(row)
        time.sleep(ORCHESTRATOR_DELAY)
        continue

    time.sleep(ORCHESTRATOR_DELAY)

    # ── B) SIMILARITY ENGINE (solo se routing["similarity"] == True) ──
    if pred_sim:
        # Usa esattamente i parametri che l'orchestratore ha prodotto
        k_sim     = routing.get("k_similar") or DEFAULT_SIM_K
        w_sim     = routing.get("weights_similarity") or DEFAULT_SIM_WEIGHTS
        # Sicurezza: se weights non è una lista di 3 float validi, usa default
        if not (isinstance(w_sim, list) and len(w_sim) == 3 and sum(w_sim) > 0):
            w_sim = DEFAULT_SIM_WEIGHTS

        target_diags = diag_dict.get(pat_int, set())

        if not target_diags or pat_int not in hadm_to_idx:
            print(f"           Similarity:    ⏭ skipped (nessuna diagnosi / non in indice)")
            row.update({
                "sim_performed": False, "sim_hit": None, "sim_mrr": None,
                "sim_best_jaccard": None, "sim_k_used": k_sim,
                "sim_weights_used": str(w_sim),
                "sim_status": "skipped (no diagnoses or index)",
            })
        else:
            try:
                t0 = time.perf_counter()
                print(f"           Weights usati: {w_sim}")
                retrieved = engine.find_similar(pat_int, k=k_sim, weights=w_sim)
                sim_elapsed = time.perf_counter() - t0

                hit_found    = False
                mrr_val      = 0.0
                best_jaccard = 0.0

                for rank, (rec_hadm, _score) in enumerate(retrieved, start=1):
                    rec_diags = diag_dict.get(int(rec_hadm), set())
                    print("TARGET", target_diags)
                    print("REC", rec_diags)
                    if not rec_diags:
                        continue
                    inter   = len(target_diags & rec_diags)
                    union   = len(target_diags | rec_diags)
                    jaccard = inter / union if union > 0 else 0.0
                    best_jaccard = max(best_jaccard, jaccard)
                    if jaccard >= JACCARD_THRESHOLD and not hit_found:
                        hit_found = True
                        mrr_val   = 1.0 / rank

                sim_icon = "✅" if hit_found else "❌"
                print(
                    f"           Similarity:    {sim_icon} {sim_elapsed:.1f}s | "
                    f"k={k_sim} | hit={hit_found} | jaccard={best_jaccard:.3f} | "
                    f"mrr={mrr_val:.3f}"
                )

                row.update({
                    "sim_performed":   True,
                    "sim_hit":         hit_found,
                    "sim_mrr":         mrr_val,
                    "sim_best_jaccard": best_jaccard,
                    "sim_k_used":      k_sim,
                    "sim_weights_used": str(w_sim),
                    "sim_status":      "success",
                })

            except Exception as e:
                print(f"           Similarity:    ❌ ERRORE: {e}")
                row.update({
                    "sim_performed": False, "sim_hit": None, "sim_mrr": None,
                    "sim_best_jaccard": None, "sim_k_used": k_sim,
                    "sim_weights_used": str(w_sim),
                    "sim_status": f"error: {e}",
                })
    else:
        print(f"           Similarity:    ⏭ non richiesta dal routing")
        row.update({
            "sim_performed": False, "sim_hit": None, "sim_mrr": None,
            "sim_best_jaccard": None, "sim_k_used": None,
            "sim_weights_used": None, "sim_status": "not requested",
        })

    rows.append(row)

engine.close()

# ── 5. Aggregazione e report finale ───────────────────────────────
df_results = pd.DataFrame(rows)

# Salva CSV completo
out_csv = f"integrated_eval_{datetime.now().strftime('%Y%m%d_%H%M%S')}.csv"
df_results.to_csv(out_csv, index=False)

# Metriche orchestratore (solo righe con successo)
orch_ok  = df_results[df_results["orch_status"] == "success"]
orch_ok  = orch_ok.copy()
orch_ok["exact_match"] = orch_ok["topics_ok"] & orch_ok["sim_ok"] & orch_ok["attrs_ok"]

# Metriche similarity (solo righe dove è stata eseguita con successo)
sim_done = df_results[df_results["sim_status"] == "success"].copy()
n_sim    = len(sim_done)
hit_rate = sim_done["sim_hit"].mean()       if n_sim > 0 else float("nan")
mrr_mean = sim_done["sim_mrr"].mean()       if n_sim > 0 else float("nan")
jac_mean = sim_done["sim_best_jaccard"].mean() if n_sim > 0 else float("nan")

print(f"\n{'='*70}")
print("  📊 REPORT FINALE — VALUTAZIONE INTEGRATA (50 coppie paziente×domanda)")
print(f"{'='*70}")
print()
print("  ── ORCHESTRATORE ─────────────────────────────────────────────")
print(f"  Coppie valutate:       {len(orch_ok)} / {N_PATIENTS}")
print(f"  Topics  Accuracy:      {orch_ok['topics_ok'].mean()*100:.1f}%")
print(f"  Sim     Accuracy:      {orch_ok['sim_ok'].mean()*100:.1f}%")
print(f"  Attrs   Accuracy:      {orch_ok['attrs_ok'].mean()*100:.1f}%")
print(f"  Score medio:           {orch_ok['orch_score'].mean()*100:.1f}%")
print(f"  Exact Match Ratio:     {orch_ok['exact_match'].mean()*100:.1f}%")
print(f"  Tempo medio:           {orch_ok['orch_time_s'].mean():.2f}s")
print()
print("  Breakdown per categoria:")
cat_agg = orch_ok.groupby("q_category").agg(
    n=("orch_score", "count"),
    topics_acc=("topics_ok", "mean"),
    sim_acc=("sim_ok", "mean"),
    attrs_acc=("attrs_ok", "mean"),
    avg_score=("orch_score", "mean"),
).round(3)
cat_agg[["topics_acc", "sim_acc", "attrs_acc", "avg_score"]] *= 100
print(cat_agg.to_string())
print()
print("  ── SIMILARITY ENGINE (Milvus ANN + Re-ranking) ───────────────")
n_requested = int(df_results["sim_performed"].notna().sum())
print(f"  Richieste dal routing: {int((df_results['pred_similarity']==True).sum())} / {N_PATIENTS}")
print(f"  Eseguite con successo: {n_sim}")
if n_sim > 0:
    print(f"  Hit-Rate @ K:          {hit_rate:.4f}  ({hit_rate*100:.1f}%)")
    print(f"  MRR @ K:               {mrr_mean:.4f}")
    print(f"  Avg Best Jaccard:      {jac_mean:.4f}")
print(f"{'='*70}")
print(f"\n  📁 CSV completo: {out_csv}")

display(df_results[[
    "hadm_id", "q_id", "q_category",
    "topics_ok", "sim_ok", "attrs_ok", "orch_score",
    "sim_performed", "sim_hit", "sim_mrr", "sim_best_jaccard",
]].head(15))


Caricamento hadm_id dal parquet...
✅ Campionati 50 pazienti (seed=42)
   Esempio ID: [25421206, 28336030, 28378969, 27924674, 24995695]

Caricamento ground truth diagnosi dal parquet...
✅ Ground truth caricata per 463002 pazienti
Connessione a Milvus e caricamento embedding...
✅ SimilarityEngine pronto

  VALUTAZIONE INTEGRATA — 50 coppie (paziente, domanda)

  [01/50] Q42 (mixed) | hadm_id=25421206
           Q: Cosa suggeriscono i dati dei pazienti simili riguardo al trattame...
           Orchestratore: ✅ 9.4s | score=1.00
[0.4, 0.3, 0.3]
TARGET {'A04', 'E78', 'G47', 'K21', 'E11', 'E03', 'N17', 'Z94', 'Y92', 'I25', 'E86', 'N18', 'K52', 'Z79', 'I13', 'T45', 'Z95'}
REC {'E78', 'Z85', 'Z87', 'F31', 'E87', 'R82', 'B18', 'Z94', 'E86', 'I10', 'M10', 'A08'}
TARGET {'A04', 'E78', 'G47', 'K21', 'E11', 'E03', 'N17', 'Z94', 'Y92', 'I25', 'E86', 'N18', 'K52', 'Z79', 'I13', 'T45', 'Z95'}
REC {'Z85', 'B18', 'I50', 'N18', 'K29', 'I48', 'E55', 'I42', 'D50', 'Z94', 'I31', 'Z79', 'E80', 'N17', 'I13',

,hadm_id,q_id,q_category,topics_ok,sim_ok,attrs_ok,orch_score,sim_performed,sim_hit,sim_mrr,sim_best_jaccard
0,25421206,Q42,mixed,True,True,True,1.000000,True,True,0.500000,0.416667
1,28336030,Q06,info_extraction,True,True,True,1.000000,False,None,NaN,NaN
2,28378969,Q02,info_extraction,True,True,True,1.000000,False,None,NaN,NaN
3,27924674,Q34,similarity,True,True,True,1.000000,True,True,0.250000,0.259259
4,24995695,Q30,pharmacology,True,True,True,1.000000,False,None,NaN,NaN
5,26299938,Q43,mixed,True,True,True,1.000000,True,True,1.000000,0.833333
6,23809725,Q13,comorbidity,True,True,True,1.000000,True,True,0.500000,0.380952
7,27268641,Q41,mixed,True,True,True,1.000000,False,None,NaN,NaN
8,23145135,Q29,pharmacology,True,False,True,0.666667,True,True,0.125000,0.200000
9,21635924,Q14,comorbidity,True,True,False,0.666667,True,True,0.500000,0.300000


## ORCHESTRATOR + SIMILARITY TEST ##

In [11]:
# ═══════════════════════════════════════════════════════════════════
# CELLA 3 — Valutazione Integrata: 50 Pazienti × 1 Domanda
#
# Sequenza per ogni coppia (hadm_id, domanda):
#   1) Chiama l'orchestratore → routing JSON
#   2) Se routing["similarity"] == True:
#        usa routing["k_similar"] e routing["weights_similarity"]
#        per chiamare SimilarityEngine.find_similar()
#        e calcola Hit-Rate / MRR (Jaccard)
#   3) Registra i risultati di entrambi i test
#
# PREREQUISITI: le celle precedenti devono essere state eseguite:
#   → definiscono CLINICAL_QUESTIONS, call_orchestrator,
#     VALID_TOPICS, VALID_ATTRS, DEFAULT_WEIGHTS
# ═══════════════════════════════════════════════════════════════════
import sys
import ast
import time
import logging
import pandas as pd
import numpy as np
from pathlib import Path
from datetime import datetime
from tqdm.auto import tqdm

# ── 0. Path setup — usa la copia locale degli agents ───────────────
AI_AGENTIC_PATH = Path("../ai-agentic").resolve()
RAGAS_PATH      = Path(".").resolve()  # cartella corrente (ai-agentic-ragas)

# Aggiungi prima la cartella ragas (ha agents/ e config/ locali)
for p in [str(RAGAS_PATH), str(AI_AGENTIC_PATH)]:
    if p not in sys.path:
        sys.path.insert(0, p)

from config.settings import Settings
from agents.similarity_engine import SimilarityEngine

# Silenzia logger interni dell'engine
for _lg in ("similarity_engine", "timer", "data_gatherer"):
    logging.getLogger(_lg).setLevel(logging.WARNING)

# ── 1. Parametri ───────────────────────────────────────────────────
CHECKPOINT_DIR    = AI_AGENTIC_PATH / "data" / "pipeline_checkpoints"
PARQUET_PATH      = str(CHECKPOINT_DIR / "combined_data_slim.parquet")
N_PATIENTS        = 50
RANDOM_SEED       = 42
ORCHESTRATOR_MODEL = "gemma4:e2b"   # modello Ollama locale
ORCHESTRATOR_DELAY = 0.3
# Parametri similarity (usati come fallback se il routing non li fornisce)
DEFAULT_SIM_K         = 5
DEFAULT_SIM_WEIGHTS   = [0.60, 0.20, 0.20]  # [diag, numeric, categorical]
JACCARD_THRESHOLD     = 0.2
USE_CATEGORIES        = True   # True = categorie ICD, False = codici esatti

# ── 2. Campiona 50 hadm_id casuali dal parquet ─────────────────────
print("Caricamento hadm_id dal parquet...")
df_ids = pd.read_parquet(PARQUET_PATH, columns=["hadm_id"])
all_hadm_ids = df_ids["hadm_id"].to_numpy()

rng = np.random.default_rng(RANDOM_SEED)
sample_patients = rng.choice(all_hadm_ids, size=N_PATIENTS, replace=False).tolist()
print(f"✅ Campionati {N_PATIENTS} pazienti (seed={RANDOM_SEED})")
print(f"   Esempio ID: {[int(x) for x in sample_patients[:5]]}\n")

# Assegna una domanda casuale a ciascun paziente (shuffle delle 50 domande)
rng2 = np.random.default_rng(RANDOM_SEED + 1)
question_order = rng2.permutation(len(CLINICAL_QUESTIONS)).tolist()
assigned_questions = [CLINICAL_QUESTIONS[question_order[i]] for i in range(N_PATIENTS)]

# ── 3. Inizializza SimilarityEngine UNA SOLA VOLTA ────────────────
print("Caricamento ground truth diagnosi dal parquet...")
col_name = "diagnosis_categories_list" if USE_CATEGORIES else "diagnoses"
df_diag  = pd.read_parquet(PARQUET_PATH, columns=["hadm_id", col_name])

diag_dict: dict = {}
for _, row in df_diag.iterrows():
    val = row[col_name]
    hid = int(row["hadm_id"])
    
    # Estraiamo i codici e tagliamoli alla terza cifra
    raw_set = set()
    if isinstance(val, (list, np.ndarray)):
        raw_set = set(val)
    elif isinstance(val, str):
        try:
            raw_set = set(ast.literal_eval(val))
        except Exception:
            raw_set = set(val.split(","))
    
    # AGGIUNGI QUESTA RIGA: Tronca ogni codice a 3 caratteri
    diag_dict[hid] = {str(code)[:3] for code in raw_set if code}
print(f"✅ Ground truth caricata per {len(diag_dict)} pazienti")

# hadm_to_idx: mappa hadm_id → indice posizionale (ordine nel parquet = ordine .npy)
hadm_to_idx = {int(hid): i for i, hid in enumerate(all_hadm_ids)}

settings = Settings()
settings.checkpoint_path = str(CHECKPOINT_DIR)
engine = SimilarityEngine(settings)
print("Connessione a Milvus e caricamento embedding...")
engine.load(hadm_to_idx)
print("✅ SimilarityEngine pronto\n")

# ── 4. Loop principale: 1 paziente → 1 domanda ────────────────────
print("=" * 70)
print(f"  VALUTAZIONE INTEGRATA — {N_PATIENTS} coppie (paziente, domanda)")
print("=" * 70)

rows = []

for i, (patient_id, question) in enumerate(
    zip(sample_patients, assigned_questions), start=1
):
    pat_int = int(patient_id)
    qid     = question["id"]
    qcat    = question["category"]
    qtext   = question["text"]

    print(f"\n  [{i:02d}/{N_PATIENTS}] {qid} ({qcat}) | hadm_id={pat_int}")
    print(f"           Q: {qtext[:65]}...")

    row = {
        "patient_idx":    i,
        "hadm_id":        pat_int,
        "q_id":           qid,
        "q_category":     qcat,
        "q_text":         qtext,
        "gt_similarity":  question["gt_similarity"],
        "gt_topics":      question["gt_topics"],
        "gt_has_attrs":   question["gt_has_attrs"],
    }

    # ── A) ORCHESTRATORE ──────────────────────────────────────────
    try:
        t0 = time.perf_counter()
        routing = call_orchestrator(qtext, pat_int, ORCHESTRATOR_MODEL)
        orch_elapsed = time.perf_counter() - t0

        pred_topics = set(routing.get("topics", []))
        gt_topics   = set(question["gt_topics"])
        topics_ok   = gt_topics.issubset(pred_topics)

        pred_sim = routing.get("similarity", False)
        gt_sim   = question["gt_similarity"]
        sim_ok   = (pred_sim == gt_sim)

        pred_attrs = set(routing.get("attributes_list", []))
        gt_attrs   = set(question["gt_has_attrs"])
        attrs_ok   = gt_attrs.issubset(pred_attrs)

        orch_score = (topics_ok + sim_ok + attrs_ok) / 3.0
        icon = "✅" if orch_score == 1.0 else ("⚠️" if orch_score >= 0.66 else "❌")

        print(f"           Orchestratore: {icon} {orch_elapsed:.1f}s | score={orch_score:.2f}", end="")
        if not topics_ok:
            print(f" | topics attesi={question['gt_topics']} ottenuti={list(pred_topics)}", end="")
        if not sim_ok:
            print(f" | sim atteso={gt_sim} ottenuto={pred_sim}", end="")
        if not attrs_ok:
            print(f" | attrs mancanti={gt_attrs - pred_attrs}", end="")
        print()

        row.update({
            "pred_topics":     list(pred_topics),
            "pred_similarity": pred_sim,
            "pred_k_similar":  routing.get("k_similar", 0),
            "pred_weights":    routing.get("weights_similarity"),
            "pred_attrs":      list(pred_attrs),
            "pred_reasoning":  routing.get("reasoning", ""),
            "topics_ok":       topics_ok,
            "sim_ok":          sim_ok,
            "attrs_ok":        attrs_ok,
            "orch_score":      orch_score,
            "orch_time_s":     round(orch_elapsed, 2),
            "orch_status":     "success",
        })

    except Exception as e:
        print(f"           Orchestratore: ❌ ERRORE: {e}")
        row.update({
            "pred_topics": [], "pred_similarity": None,
            "pred_k_similar": None, "pred_weights": None,
            "pred_attrs": [], "pred_reasoning": "",
            "topics_ok": False, "sim_ok": False, "attrs_ok": False,
            "orch_score": 0.0, "orch_time_s": 0.0,
            "orch_status": f"error: {e}",
        })
        # Senza routing valido salta la similarity
        row.update({
            "sim_performed": False, "sim_hit": None,
            "sim_mrr": None, "sim_best_jaccard": None,
            "sim_k_used": None, "sim_weights_used": None,
            "sim_status": "skipped (orch error)",
        })
        rows.append(row)
        time.sleep(ORCHESTRATOR_DELAY)
        continue

    time.sleep(ORCHESTRATOR_DELAY)

    # ── B) SIMILARITY ENGINE (solo se routing["similarity"] == True) ──
    if pred_sim:
        # Usa esattamente i parametri che l'orchestratore ha prodotto
        k_sim     = routing.get("k_similar") or DEFAULT_SIM_K
        w_sim     = routing.get("weights_similarity") or DEFAULT_SIM_WEIGHTS
        # Sicurezza: se weights non è una lista di 3 float validi, usa default
        if not (isinstance(w_sim, list) and len(w_sim) == 3 and sum(w_sim) > 0):
            w_sim = DEFAULT_SIM_WEIGHTS

        target_diags = diag_dict.get(pat_int, set())

        if not target_diags or pat_int not in hadm_to_idx:
            print(f"           Similarity:    ⏭ skipped (nessuna diagnosi / non in indice)")
            row.update({
                "sim_performed": False, "sim_hit": None, "sim_mrr": None,
                "sim_best_jaccard": None, "sim_k_used": k_sim,
                "sim_weights_used": str(w_sim),
                "sim_status": "skipped (no diagnoses or index)",
            })
        else:
            try:
                t0 = time.perf_counter()
                print(f"           Weights usati: {w_sim}")
                retrieved = engine.find_similar(pat_int, k=k_sim, weights=w_sim)
                sim_elapsed = time.perf_counter() - t0

                hit_found    = False
                mrr_val      = 0.0
                best_jaccard = 0.0
                
                # --- NUOVE VARIABILI PER METRICHE AVANZATE ---
                hits_count = 0
                cumulative_jaccard = 0.0
                jaccard_scores = [] # Lista per calcolare l'Ideal DCG

                for rank, (rec_hadm, _score) in enumerate(retrieved, start=1):
                    rec_diags = diag_dict.get(int(rec_hadm), set())
                    
                    if not rec_diags:
                        jaccard_scores.append(0.0) # Nessuna diagnosi = 0 score
                        continue
                        
                    inter   = len(target_diags & rec_diags)
                    union   = len(target_diags | rec_diags)
                    jaccard = inter / union if union > 0 else 0.0
                    
                    # Accumulo dei dati
                    best_jaccard = max(best_jaccard, jaccard)
                    jaccard_scores.append(jaccard)
                    cumulative_jaccard += jaccard
                    
                    # Logica Hit e MRR (mantenuta per confronto storico)
                    if jaccard >= JACCARD_THRESHOLD:
                        hits_count += 1
                        if not hit_found:
                            hit_found = True
                            mrr_val   = 1.0 / rank

                # --- CALCOLO nDCG, PRECISION E AVG JACCARD ---
                precision_at_k = hits_count / k_sim if k_sim > 0 else 0.0
                avg_jaccard    = cumulative_jaccard / k_sim if k_sim > 0 else 0.0
                
                # 1. Calcolo del DCG effettivo
                # Formula: somma( rel_i / log2(rank_i + 1) )
                dcg = sum(rel / np.log2(rank + 1) for rank, rel in enumerate(jaccard_scores, start=1))
                
                # 2. Calcolo dell'Ideal DCG (IDCG) ordinando dal migliore al peggiore
                ideal_scores = sorted(jaccard_scores, reverse=True)
                idcg = sum(rel / np.log2(rank + 1) for rank, rel in enumerate(ideal_scores, start=1))
                
                # 3. Normalizzazione finale
                ndcg = (dcg / idcg) if idcg > 0 else 0.0

                sim_icon = "✅" if hit_found else "❌"
                print(
                    f"           Similarity:    {sim_icon} {sim_elapsed:.1f}s | "
                    f"k={k_sim} | prec={precision_at_k:.2f} | ndcg={ndcg:.3f} | "
                    f"mrr={mrr_val:.3f}"
                )

                row.update({
                    "sim_performed":    True,
                    "sim_hit":          hit_found,
                    "sim_mrr":          mrr_val,
                    "sim_best_jaccard": best_jaccard,
                    "sim_avg_jaccard":  avg_jaccard,       # NUOVO
                    "sim_precision":    precision_at_k,    # NUOVO
                    "sim_ndcg":         ndcg,              # NUOVO
                    "sim_k_used":       k_sim,
                    "sim_weights_used": str(w_sim),
                    "sim_status":       "success",
                })

            except Exception as e:
                print(f"           Similarity:    ❌ ERRORE: {e}")
                row.update({
                    "sim_performed": False, "sim_hit": None, "sim_mrr": None,
                    "sim_best_jaccard": None, "sim_avg_jaccard": None,
                    "sim_precision": None, "sim_ndcg": None, 
                    "sim_k_used": k_sim, "sim_weights_used": str(w_sim),
                    "sim_status": f"error: {e}",
                })
    else:
        print(f"           Similarity:    ⏭ non richiesta dal routing")
        row.update({
            "sim_performed": False, "sim_hit": None, "sim_mrr": None,
            "sim_best_jaccard": None, "sim_avg_jaccard": None,
            "sim_precision": None, "sim_ndcg": None,
            "sim_k_used": None, "sim_weights_used": None, 
            "sim_status": "not requested",
        })

    rows.append(row)

engine.close()

# ── 5. Aggregazione e report finale ───────────────────────────────
df_results = pd.DataFrame(rows)

# Salva CSV completo
out_csv = f"integrated_eval_{datetime.now().strftime('%Y%m%d_%H%M%S')}.csv"
df_results.to_csv(out_csv, index=False)

# Metriche orchestratore (solo righe con successo)
orch_ok  = df_results[df_results["orch_status"] == "success"]
orch_ok  = orch_ok.copy()
orch_ok["exact_match"] = orch_ok["topics_ok"] & orch_ok["sim_ok"] & orch_ok["attrs_ok"]

# Metriche similarity (solo righe dove è stata eseguita con successo)
sim_done = df_results[df_results["sim_status"] == "success"].copy()
n_sim    = len(sim_done)

hit_rate  = sim_done["sim_hit"].mean()       if n_sim > 0 else float("nan")
mrr_mean  = sim_done["sim_mrr"].mean()       if n_sim > 0 else float("nan")
jac_mean  = sim_done["sim_best_jaccard"].mean() if n_sim > 0 else float("nan")

# Estrazione delle nuove metriche
avg_jac_mean = sim_done["sim_avg_jaccard"].mean() if n_sim > 0 else float("nan")
prec_mean    = sim_done["sim_precision"].mean()   if n_sim > 0 else float("nan")
ndcg_mean    = sim_done["sim_ndcg"].mean()        if n_sim > 0 else float("nan")

print(f"\n{'='*70}")
print("  📊 REPORT FINALE — VALUTAZIONE INTEGRATA (50 coppie paziente×domanda)")
print(f"{'='*70}")
print()
print("  ── ORCHESTRATORE ─────────────────────────────────────────────")
print(f"  Coppie valutate:       {len(orch_ok)} / {N_PATIENTS}")
print(f"  Topics  Accuracy:      {orch_ok['topics_ok'].mean()*100:.1f}%")
print(f"  Sim     Accuracy:      {orch_ok['sim_ok'].mean()*100:.1f}%")
print(f"  Attrs   Accuracy:      {orch_ok['attrs_ok'].mean()*100:.1f}%")
print(f"  Score medio:           {orch_ok['orch_score'].mean()*100:.1f}%")
print(f"  Exact Match Ratio:     {orch_ok['exact_match'].mean()*100:.1f}%")
print(f"  Tempo medio:           {orch_ok['orch_time_s'].mean():.2f}s")
print()
print("  Breakdown per categoria:")
cat_agg = orch_ok.groupby("q_category").agg(
    n=("orch_score", "count"),
    topics_acc=("topics_ok", "mean"),
    sim_acc=("sim_ok", "mean"),
    attrs_acc=("attrs_ok", "mean"),
    avg_score=("orch_score", "mean"),
).round(3)
cat_agg[["topics_acc", "sim_acc", "attrs_acc", "avg_score"]] *= 100
print(cat_agg.to_string())
print()
print("  ── SIMILARITY ENGINE (Milvus ANN + Re-ranking) ───────────────")
n_requested = int(df_results["sim_performed"].notna().sum())
print(f"  Richieste dal routing: {int((df_results['pred_similarity']==True).sum())} / {N_PATIENTS}")
print(f"  Eseguite con successo: {n_sim}")
if n_sim > 0:
    print(f"  Hit-Rate @ K:          {hit_rate:.4f}  ({hit_rate*100:.1f}%)")
    print(f"  MRR @ K:               {mrr_mean:.4f}  <-- (Penalizza i rank bassi)")
    print(f"  nDCG @ K:              {ndcg_mean:.4f}  <-- (Premia la qualità globale)")
    print(f"  Precision @ K:         {prec_mean:.4f}  <-- (Densità media di Hit)")
    print(f"  Avg Best Jaccard:      {jac_mean:.4f}")
    print(f"  Avg Context Jaccard:   {avg_jac_mean:.4f}")
print(f"{'='*70}")
print(f"\n  📁 CSV completo: {out_csv}")

display(df_results[[
    "hadm_id", "q_id", "orch_score",
    "sim_performed", "sim_hit", "sim_precision", "sim_ndcg", "sim_avg_jaccard"
]].head(50))

Caricamento hadm_id dal parquet...
✅ Campionati 50 pazienti (seed=42)
   Esempio ID: [25421206, 28336030, 28378969, 27924674, 24995695]

Caricamento ground truth diagnosi dal parquet...
✅ Ground truth caricata per 463002 pazienti
Connessione a Milvus e caricamento embedding...
✅ SimilarityEngine pronto

  VALUTAZIONE INTEGRATA — 50 coppie (paziente, domanda)

  [01/50] Q42 (mixed) | hadm_id=25421206
           Q: Cosa suggeriscono i dati dei pazienti simili riguardo al trattame...
           Orchestratore: ✅ 13.3s | score=1.00
           Weights usati: [0.4, 0.3, 0.3]
           Similarity:    ✅ 0.1s | k=10 | prec=0.70 | ndcg=0.795 | mrr=0.500

  [02/50] Q06 (info_extraction) | hadm_id=28336030
           Q: Quali sono i dettagli di ammissione di questo ricovero?...
           Orchestratore: ✅ 4.0s | score=1.00
           Similarity:    ⏭ non richiesta dal routing

  [03/50] Q02 (info_extraction) | hadm_id=28378969
           Q: Estrai i dati anagrafici principali del paziente dal data

,hadm_id,q_id,orch_score,sim_performed,sim_hit,sim_precision,sim_ndcg,sim_avg_jaccard
0,25421206,Q42,1.000000,True,True,0.7,0.794635,0.257278
1,28336030,Q06,1.000000,False,None,NaN,NaN,NaN
2,28378969,Q02,1.000000,False,None,NaN,NaN,NaN
3,27924674,Q34,1.000000,True,True,0.2,0.826957,0.152936
4,24995695,Q30,1.000000,False,None,NaN,NaN,NaN
5,26299938,Q43,1.000000,True,True,1.0,0.980871,0.655198
6,23809725,Q13,1.000000,True,True,0.6,0.872296,0.220253
7,27268641,Q41,1.000000,False,None,NaN,NaN,NaN
8,23145135,Q29,0.666667,True,True,0.1,0.667307,0.086106
9,21635924,Q14,0.666667,True,True,0.3,0.745696,0.146831


## JACCARD THREESHOLD 0.2, 0.3, 0.4, 0.5 ##

In [1]:
# ═══════════════════════════════════════════════════════════════════
# CELLA 4 — Analisi della Sensibilità alle Soglie Jaccard
#
# Questa cella ricalcola Hit-Rate, Precision e MRR per diverse soglie 
# di Jaccard (0.2, 0.3, 0.4, 0.5) riutilizzando i pazienti e i parametri
# salvati nell'ultimo CSV di valutazione integrata generato.
# ═══════════════════════════════════════════════════════════════════
import os
import glob
import ast
import pandas as pd
import numpy as np
from pathlib import Path
from config.settings import Settings
from agents.similarity_engine import SimilarityEngine

# 1. Trova l'ultimo CSV di valutazione generato
csv_files = glob.glob("integrated_eval_*.csv")
if not csv_files:
    csv_files = glob.glob("orchestrator_eval_*.csv")

if not csv_files:
    print("❌ Nessun file CSV di valutazione trovato. Esegui prima la Cella 3 per generare i risultati.")
else:
    latest_csv = max(csv_files, key=os.path.getctime)
    print(f"📁 Caricamento dati da: {latest_csv}")
    df_eval = pd.read_csv(latest_csv)
    
    sim_cases = df_eval[df_eval["sim_status"] == "success"].copy()
    print(f"🔍 Trovati {len(sim_cases)} casi con ricerca di similarità eseguita.")
    
    if len(sim_cases) == 0:
        print("❌ Nessun caso di similarity con successo trovato nel CSV.")
    else:
        AI_AGENTIC_PATH = Path("../ai-agentic").resolve()
        CHECKPOINT_DIR    = AI_AGENTIC_PATH / "data" / "pipeline_checkpoints"
        PARQUET_PATH      = str(CHECKPOINT_DIR / "combined_data_slim.parquet")
        USE_CATEGORIES    = True
        
        print("Caricamento ground truth diagnosi...")
        col_name = "diagnosis_categories_list" if USE_CATEGORIES else "diagnoses"
        df_diag  = pd.read_parquet(PARQUET_PATH, columns=["hadm_id", col_name])
        
        diag_dict = {}
        for _, row in df_diag.iterrows():
            val = row[col_name]
            hid = int(row["hadm_id"])
            raw_set = set()
            if isinstance(val, (list, np.ndarray)):
                raw_set = set(val)
            elif isinstance(val, str):
                try:
                    raw_set = set(ast.literal_eval(val))
                except Exception:
                    raw_set = set(val.split(","))
            diag_dict[hid] = {str(code)[:3] for code in raw_set if code}
            
        df_ids = pd.read_parquet(PARQUET_PATH, columns=["hadm_id"])
        all_hadm_ids = df_ids["hadm_id"].to_numpy()
        hadm_to_idx = {int(hid): i for i, hid in enumerate(all_hadm_ids)}
        
        settings = Settings()
        settings.checkpoint_path = str(CHECKPOINT_DIR)
        engine = SimilarityEngine(settings)
        engine.load(hadm_to_idx)
        
        thresholds = [0.2, 0.3, 0.4, 0.5]
        threshold_results = {t: {"hits": [], "precisions": [], "mrrs": []} for t in thresholds}
        
        print("Ricalcolo similarità in corso...")
        for _, row in sim_cases.iterrows():
            pat_int = int(row["hadm_id"])
            k_sim = int(row["sim_k_used"])
            w_str = row["sim_weights_used"]
            w_sim = ast.literal_eval(w_str)
            
            target_diags = diag_dict.get(pat_int, set())
            if not target_diags:
                continue
                
            retrieved = engine.find_similar(pat_int, k=k_sim, weights=w_sim)
            
            jaccard_scores = []
            for rec_hadm, _ in retrieved:
                rec_diags = diag_dict.get(int(rec_hadm), set())
                if not rec_diags:
                    jaccard_scores.append(0.0)
                    continue
                inter = len(target_diags & rec_diags)
                union = len(target_diags | rec_diags)
                jaccard = inter / union if union > 0 else 0.0
                jaccard_scores.append(jaccard)
                
            for t in thresholds:
                hit = any(j >= t for j in jaccard_scores)
                threshold_results[t]["hits"].append(hit)
                
                hits_count = sum(1 for j in jaccard_scores if j >= t)
                precision = hits_count / k_sim if k_sim > 0 else 0.0
                threshold_results[t]["precisions"].append(precision)
                
                mrr = 0.0
                for rank, j in enumerate(jaccard_scores, start=1):
                    if j >= t:
                        mrr = 1.0 / rank
                        break
                threshold_results[t]["mrrs"].append(mrr)
                
        engine.close()
        
        summary_rows = []
        for t in thresholds:
            avg_hit = np.mean(threshold_results[t]["hits"]) * 100
            avg_prec = np.mean(threshold_results[t]["precisions"]) * 100
            avg_mrr = np.mean(threshold_results[t]["mrrs"])
            summary_rows.append({
                "Soglia Jaccard": t,
                "Hit-Rate (%)": f"{avg_hit:.1f}%",
                "Precision@K (Pertinenza Media %)": f"{avg_prec:.1f}%",
                "MRR medio": f"{avg_mrr:.4f}"
            })
            
        df_summary = pd.DataFrame(summary_rows)
        print("\n" + "="*70)
        print("  📊 CONFRONTO METRICHE RETRIEVAL PER DIVERSE SOGLIE JACCARD")
        print("="*70)
        print(df_summary.to_string(index=False))
        print("="*70)


📁 Caricamento dati da: integrated_eval_20260515_124147.csv
🔍 Trovati 35 casi con ricerca di similarità eseguita.
Caricamento ground truth diagnosi...


[06/03/26 16:49:06] INFO     ▶ SimilarityEngine — Caricamento

                    INFO     Connessione a Milvus (localhost:19530)...

[06/03/26 16:49:25] INFO     Connesso a Milvus.

[06/03/26 16:49:26] INFO     Embedding caricati — numeric: (463002, 52), categorical: (463002, 8), diagnosis:      
                             (463002, 100)

[06/03/26 16:49:46] INFO     Collection Milvus 'mimiciv_diagnosis_embeddings': 463002 entità

... (140 outputs left out)

                    INFO     FASE 2: Re-ranking completato, top 10 selezionati

                    INFO     ◼ Ricerca 10 pazienti simili (pool=200) completato in 0.04s

                    INFO     Connessione Milvus chiusa.


  📊 CONFRONTO METRICHE RETRIEVAL PER DIVERSE SOGLIE JACCARD
 Soglia Jaccard Hit-Rate (%) Precision@K (Pertinenza Media %) MRR medio
            0.2        88.6%                            49.1%    0.6179
            0.3        71.4%                            27.4%    0.4593
            0.4        42.9%                            18.3%    0.3008
            0.5        28.6%                            14.3%    0.2362


## SIMILARITY FASTEXT+TF-IDF VS TF-IDF ONLY

In [1]:
# ═══════════════════════════════════════════════════════════════════
# VALUTAZIONE COMPARATIVA RETRIEVAL: FastText + Milvus vs Baseline TF-IDF
# (Focus esclusivo sulle DIAGNOSI: indicizzazione a 4 char, valutazione a 3 char)
# ═══════════════════════════════════════════════════════════════════
import sys
import ast
import time
import logging
import pandas as pd
import numpy as np
from pathlib import Path
from datetime import datetime
from tqdm.auto import tqdm
from sklearn.feature_extraction.text import TfidfVectorizer

# ── 0. Path setup ──────────────────────────────────────────────────
AI_AGENTIC_PATH = Path("../ai-agentic").resolve()
RAGAS_PATH      = Path(".").resolve()

for p in [str(RAGAS_PATH), str(AI_AGENTIC_PATH)]:
    if p not in sys.path:
        sys.path.insert(0, p)

from config.settings import Settings
from agents.similarity_engine import SimilarityEngine

# Silenzia logger interni
for _lg in ("similarity_engine", "timer", "data_gatherer"):
    logging.getLogger(_lg).setLevel(logging.WARNING)

# ── 1. Parametri ───────────────────────────────────────────────────
CHECKPOINT_DIR    = AI_AGENTIC_PATH / "data" / "pipeline_checkpoints"
PARQUET_PATH      = str(CHECKPOINT_DIR / "combined_data_slim.parquet")
N_PATIENTS        = 100
RANDOM_SEED       = 42
K_EVAL            = 10
JACCARD_THRESHOLD = 0.2
DIAG_ONLY_WEIGHTS = [1.0, 0.0, 0.0]

# ── 2. Caricamento Dataset e Campionamento Pazienti ────────────────
print("Caricamento dataset dal parquet...")
df_diag = pd.read_parquet(PARQUET_PATH, columns=["hadm_id", "diagnosis_categories_list"])
all_hadm_ids = df_diag["hadm_id"].to_numpy()

rng = np.random.default_rng(RANDOM_SEED)
sample_patients = rng.choice(all_hadm_ids, size=N_PATIENTS, replace=False).tolist()
print(f"✅ Campionati {N_PATIENTS} pazienti target per il benchmark (seed={RANDOM_SEED})")

# ── 3. Vocabolario: 4 char per Retrieval, 3 char per Valutazione ───
print("Preparazione vocabolari diagnosi...")
diag_gt_3char = {}  # Ground truth a 3 char (per calcolo Jaccard)
diag_corpus_4char = []  # Corpus a 4 char (per TF-IDF Vectorizer)

hadm_raw = df_diag["hadm_id"].values
col_raw = df_diag["diagnosis_categories_list"].values

for hid_val, val in zip(hadm_raw, col_raw):
    hid = int(hid_val)
    raw_set = set()
    if isinstance(val, (list, np.ndarray)):
        raw_set = set(val)
    elif isinstance(val, str):
        try:
            raw_set = set(ast.literal_eval(val))
        except Exception:
            raw_set = set(val.split(","))
    
    # 1. Retrieval corpus (4 caratteri completi)
    codes_4char = {str(code).strip() for code in raw_set if str(code).strip() not in ['nan', 'None', '']}
    diag_corpus_4char.append(" ".join(codes_4char))
    
    # 2. Ground Truth Jaccard (macro-categorie a 3 caratteri)
    diag_gt_3char[hid] = {code[:3] for code in codes_4char if code}

hadm_to_idx = {int(hid): i for i, hid in enumerate(all_hadm_ids)}
idx_to_hadm = {i: int(hid) for i, hid in enumerate(all_hadm_ids)}
print(f"✅ Dati preparati per {len(diag_gt_3char)} ricoveri")

# ── 4. Inizializzazione SimilarityEngine (FastText + Milvus) ───────
settings = Settings()
settings.checkpoint_path = str(CHECKPOINT_DIR)
engine = SimilarityEngine(settings)
print("Connessione a Milvus e caricamento FastText...")
engine.load(hadm_to_idx)
print("✅ SimilarityEngine (FastText) pronto")

# ── 5. Inizializzazione Baseline TF-IDF (4 char, 100 features) ─────
print("Fitting TF-IDF Vectorizer (max_features=100 sui codici a 4 char)...")
t_tfidf0 = time.perf_counter()
tfidf_diag_vectorizer = TfidfVectorizer(max_features=100, token_pattern=r"(?u)\b\w+\b")
X_diag_tfidf = tfidf_diag_vectorizer.fit_transform(diag_corpus_4char)
print(f"✅ TF-IDF Matrix pronta: {X_diag_tfidf.shape} ({time.perf_counter()-t_tfidf0:.2f}s)\n")

def find_similar_tfidf_diag(target_hadm_id: int, k: int = K_EVAL):
    t_idx = hadm_to_idx.get(target_hadm_id)
    if t_idx is None:
        return []
    target_vec = X_diag_tfidf[t_idx]
    scores = X_diag_tfidf.dot(target_vec.T).toarray().ravel()
    scores[t_idx] = -1.0  # Esclude se stesso
    top_indices = np.argpartition(scores, -k)[-k:]
    top_indices = top_indices[np.argsort(-scores[top_indices])]
    return [(idx_to_hadm[idx], float(scores[idx])) for idx in top_indices]

# ── 6. Benchmark con memorizzazione vettori Jaccard (per TUTTI i pazienti) ──
print("=" * 75)
print(f"  RETRIEVAL SU TUTTI I {N_PATIENTS} PAZIENTI TARGET")
print("=" * 75)

THRESHOLDS = [0.2, 0.3, 0.4, 0.5]
records_ft = []
records_tfidf = []
ft_times = []
tfidf_times = []

for pat_id in tqdm(sample_patients, desc="Retrieval"):
    pat_int = int(pat_id)
    target_diags = diag_gt_3char.get(pat_int, set())

    # 1. FastText (Milvus)
    t0 = time.perf_counter()
    ret_ft = engine.find_similar(pat_int, k=K_EVAL, weights=DIAG_ONLY_WEIGHTS)
    ft_times.append(time.perf_counter() - t0)
    
    j_ft = []
    for rec_hadm, _ in ret_ft:
        rec_diags = diag_gt_3char.get(int(rec_hadm), set())
        inter = len(target_diags & rec_diags)
        union = len(target_diags | rec_diags)
        j_ft.append(inter / union if union > 0 else 0.0)
    records_ft.append(j_ft)

    # 2. TF-IDF (Cosine 4 char)
    t0 = time.perf_counter()
    ret_tfidf = find_similar_tfidf_diag(pat_int, k=K_EVAL)
    tfidf_times.append(time.perf_counter() - t0)
    
    j_tfidf = []
    for rec_hadm, _ in ret_tfidf:
        rec_diags = diag_gt_3char.get(int(rec_hadm), set())
        inter = len(target_diags & rec_diags)
        union = len(target_diags | rec_diags)
        j_tfidf.append(inter / union if union > 0 else 0.0)
    records_tfidf.append(j_tfidf)

engine.close()

# ── 7. Funzione di calcolo metriche IR parametrizzata sulla soglia ──
def evaluate_for_threshold(records, threshold, k=K_EVAL):
    hits, mrrs, precisions = [], [], []
    best_jaccards, avg_jaccards, ndcgs = [], [], []

    for j_list in records:
        if not j_list:
            continue
        best_jaccards.append(max(j_list))
        avg_jaccards.append(np.mean(j_list))

        # Metriche dipendenti dalla soglia
        hit_ranks = [r for r, val in enumerate(j_list, start=1) if val >= threshold]
        hits.append(1 if len(hit_ranks) > 0 else 0)
        mrrs.append(1.0 / hit_ranks[0] if len(hit_ranks) > 0 else 0.0)
        precisions.append(len(hit_ranks) / k)

        # nDCG continuo pesato su Jaccard
        dcg = sum(val / np.log2(r + 1) for r, val in enumerate(j_list, start=1))
        ideal = sorted(j_list, reverse=True)
        idcg = sum(val / np.log2(r + 1) for r, val in enumerate(ideal, start=1))
        ndcgs.append(dcg / idcg if idcg > 0 else 0.0)

    return {
        "Hit-Rate@10": f"{np.mean(hits)*100:.1f}%",
        "Precision@10": f"{np.mean(precisions)*100:.1f}%",
        "MRR@10": f"{np.mean(mrrs):.4f}",
        "nDCG@10": f"{np.mean(ndcgs):.4f}",
        "Best Jaccard": f"{np.mean(best_jaccards):.4f}",
        "Context Jaccard": f"{np.mean(avg_jaccards):.4f}",
    }

# ── 8. Report Finale Comparativo Multi-Soglia ───────────────────────
rows_summary = []
for th in THRESHOLDS:
    m_ft = evaluate_for_threshold(records_ft, th)
    m_tfidf = evaluate_for_threshold(records_tfidf, th)
    
    rows_summary.append({
        "Soglia (Jaccard)": f"τ = {th}",
        "Metodo": "FastText + Milvus",
        "Hit-Rate@10": m_ft["Hit-Rate@10"],
        "Precision@10": m_ft["Precision@10"],
        "MRR@10": m_ft["MRR@10"],
        "nDCG@10": m_ft["nDCG@10"],
        "Best Jaccard": m_ft["Best Jaccard"],
        "Context Jaccard": m_ft["Context Jaccard"],
        "Tempo Medio": f"{np.mean(ft_times):.3f}s"
    })
    rows_summary.append({
        "Soglia (Jaccard)": f"τ = {th}",
        "Metodo": "TF-IDF 100D Cosine",
        "Hit-Rate@10": m_tfidf["Hit-Rate@10"],
        "Precision@10": m_tfidf["Precision@10"],
        "MRR@10": m_tfidf["MRR@10"],
        "nDCG@10": m_tfidf["nDCG@10"],
        "Best Jaccard": m_tfidf["Best Jaccard"],
        "Context Jaccard": m_tfidf["Context Jaccard"],
        "Tempo Medio": f"{np.mean(tfidf_times):.3f}s"
    })

df_threshold_comparison = pd.DataFrame(rows_summary)

print("\n" + "=" * 95)
print("📊 REPORT COMPARATIVO MULTI-SOGLIA (τ = 0.2, 0.3, 0.4, 0.5): FASTTEXT vs TF-IDF")
print("=" * 95)
print(df_threshold_comparison.to_string(index=False))


tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Caricamento dataset dal parquet...
✅ Campionati 100 pazienti target per il benchmark (seed=42)
Preparazione vocabolari diagnosi...
✅ Dati preparati per 463002 ricoveri
Connessione a Milvus e caricamento FastText...
✅ SimilarityEngine (FastText) pronto
Fitting TF-IDF Vectorizer (max_features=100 sui codici a 4 char)...
✅ TF-IDF Matrix pronta: (463002, 100) (5.31s)

  RETRIEVAL SU TUTTI I 100 PAZIENTI TARGET


Retrieval: 100%|██████████| 100/100 [00:08<00:00, 11.21it/s]


📊 REPORT COMPARATIVO MULTI-SOGLIA (τ = 0.2, 0.3, 0.4, 0.5): FASTTEXT vs TF-IDF
Soglia (Jaccard)             Metodo Hit-Rate@10 Precision@10 MRR@10 nDCG@10 Best Jaccard Context Jaccard Tempo Medio
         τ = 0.2  FastText + Milvus       98.0%        61.7% 0.8161  0.9203       0.4547          0.2857      0.052s
         τ = 0.2 TF-IDF 100D Cosine       93.0%        66.8% 0.8140  0.8703       0.3934          0.2557      0.037s
         τ = 0.3  FastText + Milvus       75.0%        38.0% 0.6177  0.9203       0.4547          0.2857      0.052s
         τ = 0.3 TF-IDF 100D Cosine       72.0%        33.2% 0.5000  0.8703       0.3934          0.2557      0.037s
         τ = 0.4  FastText + Milvus       54.0%        20.9% 0.4312  0.9203       0.4547          0.2857      0.052s
         τ = 0.4 TF-IDF 100D Cosine       43.0%        12.5% 0.2746  0.8703       0.3934          0.2557      0.037s
         τ = 0.5  FastText + Milvus       39.0%        14.0% 0.3044  0.9203       0.4547          0.2